# 09 Weekly Challenge

전체 프로세스: Qwen Model → LoRA/QLoRA → PTQ → GGUF, Llama.cpp

## 진행 순서
- Phase 0. Baseline 확보
- Phase 1. Fine-Tuning: LoRA vs QLoRA
- Phase 2. Post-Training Quantization (PTQ)
- Phase 3. GGUF 변환 및 Llama.cpp 추론
- 최종 정리 (표 1, 표 2)

## Phase 0. Baseline 확보

- 0-1. 환경 설정 및 라이브러리 import
- 0-2. Qwen 원본 모델 로드 (bfloat16)
- 0-3. 샘플 prompt 추론 테스트
- 0-4. Baseline metrics 기록 (perplexity, memory, latency)
- 0-5. [Empty Cache] 원본 모델 메모리 해제

In [12]:
# 0-1. 환경 설정 및 라이브러리 import
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"device: {device}")

# 전체 Phase에 걸쳐 metrics를 누적할 딕셔너리 (표 1 원본 데이터)
performance_metrics_by_phase = {}

device: cuda


In [13]:
# 0-2. Qwen 원본 모델 로드 (bfloat16)
model_name_or_path = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

original_qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    dtype=torch.bfloat16
).to(device)

original_qwen_model.eval()
print(f"model loaded: {model_name_or_path}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [14]:
# 0-3. 샘플 prompt 추론 테스트
sample_prompt = "다음 주 화요일 오후 3시에 회의 일정을 잡아줘."

input_token_ids = tokenizer(sample_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    generated_token_ids = original_qwen_model.generate(
        **input_token_ids,
        max_new_tokens=100,
        do_sample=False
    )

generated_text = tokenizer.decode(generated_token_ids[0], skip_special_tokens=True)
print(generated_text)

다음 주 화요일 오후 3시에 회의 일정을 잡아줘. 그리고 그날 저녁에는 어떤 음식을 먹어야 할지 알려주세요.
주말에 회의를 가질 예정입니다. 다음 주 화요일 오후 3시에 회의를 진행할 계획이니, 그 날의 일정을 잡아주시기 바랍니다.

1. 오전 9시: 회의 준비
2. 오후 10시: 회의 시작

그날 저녁은 무엇을 �


In [15]:
# 0-4. Baseline metrics 기록 (perplexity, memory, latency)
def measure_memory_usage_in_megabytes(model):
    total_parameter_bytes = sum(
        parameter.element_size() * parameter.numel()
        for parameter in model.parameters()
    )
    return total_parameter_bytes / (1024 ** 2)


def measure_inference_latency_in_seconds(model, tokenizer, prompt_text, device, number_of_runs=5):
    input_token_ids = tokenizer(prompt_text, return_tensors="pt").to(device)

    # warm-up (초기 실행 오버헤드 제외)
    with torch.no_grad():
        model.generate(**input_token_ids, max_new_tokens=50, do_sample=False)

    elapsed_time_list = []
    for _ in range(number_of_runs):
        start_time = time.time()
        with torch.no_grad():
            model.generate(**input_token_ids, max_new_tokens=50, do_sample=False)
        elapsed_time_list.append(time.time() - start_time)

    return sum(elapsed_time_list) / len(elapsed_time_list)


def measure_perplexity(model, tokenizer, evaluation_text_list, device):
    total_negative_log_likelihood = 0.0
    total_token_count = 0

    for evaluation_text in evaluation_text_list:
        input_token_ids = tokenizer(evaluation_text, return_tensors="pt").to(device)
        with torch.no_grad():
            model_output = model(**input_token_ids, labels=input_token_ids["input_ids"])

        token_count = input_token_ids["input_ids"].size(1)
        total_negative_log_likelihood += model_output.loss.item() * token_count
        total_token_count += token_count

    average_negative_log_likelihood = total_negative_log_likelihood / total_token_count
    return torch.exp(torch.tensor(average_negative_log_likelihood)).item()

In [16]:
evaluation_text_list = [
    "다음 주 화요일 오후 3시에 회의 일정을 잡아줘.",
    "이번 주 금요일에 잡힌 일정이 있는지 확인해줘.",
    "내일 오전 10시에 팀 미팅 일정을 추가해줘.",
    "다음 달 첫째 주에 워크숍 일정을 등록해줘.",
    "오늘 오후 6시 저녁 약속을 캘린더에 저장해줘.",
    "이번 주 수요일 일정을 모두 삭제해줘.",
    "다음 주 월요일부터 금요일까지 매일 아침 회의를 잡아줘.",
    "이번 달 마지막 주에 휴가 일정을 등록해줘.",
    "내일 오후 2시로 잡힌 미팅을 오후 4시로 변경해줘.",
    "이번 주말에 잡힌 일정이 있으면 알려줘.",
    "다음 주 목요일 점심 약속을 취소해줘.",
    "매주 화요일 오전 9시에 반복 일정을 등록해줘.",
    "이번 주 일정 중 가장 빠른 일정이 무엇인지 알려줘.",
    "다음 주 출장 일정을 캘린더에 추가해줘.",
    "오늘 저녁에 잡힌 약속 시간을 확인해줘.",
    "이번 달 중 비어있는 날짜를 찾아줘.",
    "다음 주 화요일 회의를 다른 요일로 옮겨줘.",
    "이번 주 금요일 오후 일정을 모두 보여줘.",
    "내일 오전 회의 참석자 명단을 확인해줘.",
    "다음 주에 예정된 모든 일정을 요약해줘.",
]

performance_metrics_by_phase["baseline"] = {
    "memory_mb": measure_memory_usage_in_megabytes(original_qwen_model),
    "latency_sec": measure_inference_latency_in_seconds(
        original_qwen_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        original_qwen_model, tokenizer, evaluation_text_list, device
    ),
}

print(performance_metrics_by_phase["baseline"])

{'memory_mb': 2944.4013671875, 'latency_sec': 2.1023120403289797, 'perplexity': 13.661505699157715}


In [17]:
# 0-5. [Empty Cache] 원본 모델 메모리 해제
def empty_device_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    # 둘 다 없으면 (cpu) 아무것도 하지 않음

gc.collect()
empty_device_cache()

### Phase 0 실행 현황

- 실행 환경: VSCode ↔ Colab 원격 연결, GPU 런타임 (CUDA, T4)
- device 분기 및 empty cache 함수를 CUDA/MPS/CPU 대응 버전으로 수정 후 정상 동작 확인
- Baseline 확정 (Qwen2.5-1.5B-Instruct, bfloat16 기준)
  - memory_mb: 2944.40
  - latency_sec: 2.10 (50 tokens 생성 기준)
  - perplexity: 13.66 (evaluation 문장 20개 기준)
- Phase 1(LoRA/QLoRA) 진행을 위한 baseline 값으로 확정

## Phase 1. Fine-Tuning: LoRA vs QLoRA

### 1-1. Dataset 준비 (DaySync 도메인 데이터)
### 1-2. LoRA
#### 1-2-1. 기반 모델 로드 (bfloat16)
#### 1-2-2. LoRA Config 설정 (rank, alpha, target_modules)
#### 1-2-3. 학습 실행
#### 1-2-4. LoRA metrics 기록
#### 1-2-5. [Empty Cache]
### 1-3. QLoRA
#### 1-3-1. 기반 모델 로드 (4-bit NF4, BitsAndBytesConfig)
#### 1-3-2. LoRA Config 설정 (1-2-2와 동일 설정값 재사용)
#### 1-3-3. 학습 실행
#### 1-3-4. QLoRA metrics 기록
#### 1-3-5. [Empty Cache]
### 1-4. LoRA vs QLoRA 비교 및 선택
#### 1-4-1. 비교 표 작성 (perplexity, exact match, 메모리, latency)
#### 1-4-2. 다음 Phase로 전달할 모델 선택 및 근거 기록

## Phase 2. Post-Training Quantization (PTQ)

### 2-1. 양자화 방식 결정
#### 2-1-1. Weight-only vs Full Quantization 선택 및 근거
#### 2-1-2. GPTQ vs AWQ 선택 및 근거
#### 2-1-3. Static vs Dynamic Quantization 선택 및 근거 (Activation Quantization 포함 시)
### 2-2. Calibration Data 준비 (Static 선택 시)
### 2-3. 양자화 실행
### 2-4. PTQ metrics 기록 (Baseline 대비, 직전 Phase 대비)
### 2-5. [Empty Cache]

## Phase 3. GGUF 변환 및 Llama.cpp 추론

### 3-1. Adapter Merge 여부 결정 및 실행
### 3-2. GGUF 변환
### 3-3. Llama.cpp 로드 및 추론 테스트
### 3-4. Phase 2 결과와 출력 일치 여부 확인 (변환 손실 검증)
### 3-5. GGUF metrics 기록

## 최종 정리

### 표 1. 원본 Qwen 대비 누적 비교 (Baseline / Fine-Tuned / PTQ / GGUF)
### 표 2. 단계별 순수 변화량 (표 1의 인접 행 차이, 파생 계산)
### 결론 및 mentoring 검증 항목 정리